# TB Portals - Kantipudi **A2** baseline (ALP regressor + cavity classifier)

Locked config: **whole-image cavity** (`--cavity-no-lung-crop`) + **cropped ALP**. 3 held-out countries x 3 seeds x 30 epochs. **Attach these Kaggle datasets before running:** `tb-portals-cxr-pngs`, `medsam-vit-b`.

## 0 - Clone the codebase

In [1]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + '/scripts'):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)

Cloning into '/kaggle/working/dl-project-codebase'...


repo ready at /kaggle/working/dl-project-codebase


Updating files: 100% (438/438), done.


## Install deps

In [2]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segment-anything", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 32.1 MB/s eta 0:00:00
deps installed


## Paths

Edit dataset slugs if yours differ.

In [3]:
import os
WORK           = "/kaggle/working"
REPO_DIR       = "/kaggle/working/dl-project-codebase"
DATASET        = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT  = f"{DATASET}/kaggle_export"
MEDSAM_CKPT    = "/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
LUNG_DECODER   = f"{REPO_DIR}/checkpoints/component4/component4_mask_decoder.pt"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
CROPS_DIR      = f"{WORK}/crops"
OUT_DIR        = f"{WORK}/checkpoints/paper_a2"
os.makedirs(OUT_DIR, exist_ok=True)
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("MEDSAM_CKPT:  ", MEDSAM_CKPT, "->", os.path.isfile(MEDSAM_CKPT))
print("LUNG_DECODER: ", LUNG_DECODER, "->", os.path.isfile(LUNG_DECODER))

KAGGLE_EXPORT: /kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs/kaggle_export -> True
MEDSAM_CKPT:   /kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth -> True
LUNG_DECODER:  /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt -> True


## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

In [4]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

country       need_c  have_c  need_n  have_n
Georgia          713    1298     701    1108
Belarus          254     285     798     894
Ukraine          500    1206     816    1775
Kazakhstan       159     304     240     384
Romania          143     233      77     171
Moldova          193     278     396     527
Azerbaijan         5       7      12      18
India              0       6       3      12
Paper manifest: 5010 images (target 5010) -> /kaggle/working/tbportals_manifest_paper.csv


## 2 - MedSAM lung crops (~25 min first time; idempotent)

In [5]:
import os, sys
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from cache_lung_crops import main as crops_main
argv = ['--manifest', PAPER_MANIFEST, '--out-dir', CROPS_DIR,
        '--medsam-ckpt', MEDSAM_CKPT, '--size', '224', '--pad', '32']
if os.path.isfile(LUNG_DECODER):
    argv += ['--lung-decoder-ckpt', LUNG_DECODER]
crops_main(argv)
print('crops ->', CROPS_DIR, '| count:', len(os.listdir(CROPS_DIR)))

[crops] device=cuda:Tesla T4
[crops] 0/5010 cached; generating the remaining 5010.
[crops] loaded fine-tuned lung decoder: /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt
[crops] 200/5010 (lung=200, fallback=0)
[crops] 400/5010 (lung=400, fallback=0)
[crops] 600/5010 (lung=600, fallback=0)
[crops] 800/5010 (lung=800, fallback=0)
[crops] 1000/5010 (lung=1000, fallback=0)
[crops] 1200/5010 (lung=1200, fallback=0)
[crops] 1400/5010 (lung=1400, fallback=0)
[crops] 1600/5010 (lung=1600, fallback=0)
[crops] 1800/5010 (lung=1800, fallback=0)
[crops] 2000/5010 (lung=2000, fallback=0)
[crops] 2200/5010 (lung=2200, fallback=0)
[crops] 2400/5010 (lung=2400, fallback=0)
[crops] 2600/5010 (lung=2600, fallback=0)
[crops] 2800/5010 (lung=2800, fallback=0)
[crops] 3000/5010 (lung=3000, fallback=0)
[crops] 3200/5010 (lung=3200, fallback=0)
[crops] 3400/5010 (lung=3400, fallback=0)
[crops] 3600/5010 (lung=3600, fallback=0)
[crops] 3800/5010 (lung=3800, fallback=0)
[

## 3 - Full A2 run (~2-3 h)

In [6]:
from src.training.train_baseline_paper import main as train_main
train_main(['--manifest', PAPER_MANIFEST, '--crops-dir', CROPS_DIR, '--out-dir', OUT_DIR,
            '--held-outs','Romania','Moldova','Kazakhstan','--seeds','0','1','2',
            '--epochs','30','--batch-size','60','--accum-steps','5','--num-workers','2',
            '--cavity-no-lung-crop'])

[paper-baseline] device=cuda
[paper-baseline] ALP input    = lung-crop /kaggle/working/crops
[paper-baseline] cavity input = WHOLE image
[paper-baseline][NOTE] at least one head trains on whole images (the paper crops both; whole-image cavity is an intentional ablation).

===== Romania  seed=0 =====
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[ALP] train=3832 val=958 test=220
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 162MB/s] 


  [ALP] epoch 00 train_mse=0.06088 val_mse=0.04714
  [ALP] epoch 01 train_mse=0.03957 val_mse=0.02690
  [ALP] epoch 02 train_mse=0.03400 val_mse=0.02800
  [ALP] epoch 03 train_mse=0.02855 val_mse=0.02637
  [ALP] epoch 04 train_mse=0.02360 val_mse=0.02826
  [ALP] epoch 05 train_mse=0.02549 val_mse=0.02452
  [ALP] epoch 06 train_mse=0.02099 val_mse=0.02608
  [ALP] epoch 07 train_mse=0.01923 val_mse=0.02317
  [ALP] epoch 08 train_mse=0.01865 val_mse=0.02927
  [ALP] epoch 09 train_mse=0.02055 val_mse=0.03441
  [ALP] epoch 10 train_mse=0.02051 val_mse=0.02453
  [ALP] epoch 11 train_mse=0.01957 val_mse=0.02502
  [ALP] epoch 12 train_mse=0.01633 val_mse=0.02970
  [ALP] epoch 13 train_mse=0.01547 val_mse=0.02364
  [ALP] epoch 14 train_mse=0.01586 val_mse=0.02603
  [ALP] epoch 15 train_mse=0.01477 val_mse=0.02341
  [ALP] epoch 16 train_mse=0.01259 val_mse=0.02667
  [ALP] epoch 17 train_mse=0.01170 val_mse=0.03251
  [ALP] epoch 18 train_mse=0.01186 val_mse=0.02629
  [ALP] epoch 19 train_mse=0.01

## 4 - Save outputs (download these)

In [7]:
!cd /kaggle/working && zip -j results_a2.zip checkpoints/paper_a2/results.csv tbportals_manifest_paper.csv
!cd /kaggle/working && zip -r -q checkpoints_a2.zip checkpoints/paper_a2
print("Saved: results_a2.zip, checkpoints_a2.zip")

  adding: results.csv (deflated 51%)
  adding: tbportals_manifest_paper.csv (deflated 77%)
Saved: results_a2.zip, checkpoints_a2.zip
